In [1]:
import numpy as np
from PIL import Image
import scipy
import cv2 as cv
import random

In [25]:
img = cv.imread("frame10.png")
gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)
gray = np.float32(gray)
dst = cv.cornerHarris(gray,2,3,0.04)

idx = dst>0.01*dst.max()

(img_h, img_w) = dst.shape 
corner_thr = 0.01*dst.max()

features1 = []
for i in range(0, img_h):
    for j in range(0, img_w):
        if dst[i][j] > corner_thr:#==corner!
            # img[i][j]=[0,0,255]
            features1.append((j,i))


feature = random.choice(features1)
img[feature[1]][feature[0]]=[0,255,255]
cv.circle(img, feature, 5, (0, 255, 255), 1)


cv.imshow('dst',img)
if cv.waitKey(0) & 0xff == 27:
    cv.destroyAllWindows()



In [18]:
img = cv.imread("frame11.png")
gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)
gray = np.float32(gray)
dst = cv.cornerHarris(gray,2,3,0.04)

idx = dst>0.01*dst.max()

(img_h, img_w) = dst.shape 
corner_thr = 0.01*dst.max()

features2 = []
for i in range(0, img_h):
    for j in range(0, img_w):
        if dst[i][j] > corner_thr:#==corner!
            # img[i][j]=[0,0,255]
            features2.append((j,i))


feature = random.choice(features2)
img[feature[0]][feature[1]]=[0,255,255]
cv.circle(img, feature, 5, (0, 255, 255), 1)


cv.imshow('dst',img)
if cv.waitKey(0) & 0xff == 27:
    cv.destroyAllWindows()



In [29]:
print(img.shape)

(388, 584, 3)


In [28]:
print(len(features1))
print(len(features2))  #추출된 피쳐수가 다름

1496
1458


In [26]:
print(features1)

[(325, 100), (326, 100), (319, 101), (320, 101), (321, 101), (325, 101), (326, 101), (331, 101), (332, 101), (304, 102), (305, 102), (325, 102), (331, 102), (332, 102), (298, 103), (299, 103), (304, 103), (305, 103), (332, 103), (183, 104), (298, 104), (299, 104), (300, 104), (304, 104), (183, 105), (184, 105), (188, 105), (277, 106), (271, 107), (272, 107), (273, 107), (277, 107), (337, 107), (272, 108), (273, 108), (337, 108), (337, 109), (338, 109), (192, 110), (193, 110), (204, 110), (248, 110), (249, 110), (307, 110), (338, 110), (347, 110), (192, 111), (216, 111), (249, 111), (306, 111), (307, 111), (338, 111), (347, 111), (171, 112), (172, 112), (173, 112), (211, 112), (222, 112), (307, 112), (347, 112), (348, 112), (171, 113), (172, 113), (173, 113), (181, 113), (255, 113), (172, 114), (183, 114), (236, 114), (243, 114), (250, 114), (252, 114), (253, 114), (254, 114), (255, 114), (256, 114), (257, 114), (271, 114), (301, 114), (332, 114), (333, 114), (336, 114), (337, 114), (35

In [38]:
#step 1.
(img_h, img_w) = gray.shape  # 1. 이미지 크기
patch_size = 23              # 2. 패치 크기
patch_size_half = patch_size // 2


valid_features1 = [(x, y) for (x, y) in features1 
                  if (x >= patch_size_half and 
                      x < img_w - patch_size_half and 
                      y >= patch_size_half and 
                      y < img_h - patch_size_half)]

(feat_x, feat_y) = random.choice(valid_features1) # a single corner point (x, y), boundary condition chk!
print((feat_x, feat_y) )


#step 2.
patch_size = 23
patch_size_half = patch_size//2 
W = gray[feat_y-patch_size_half:feat_y+patch_size_half+1, feat_x-patch_size_half:feat_x+patch_size_half+1]/255.0 #23x23 patch, brightness range=(0,1)


#step 3
# df_dx = conv(W, s_x)
# dw_dy = conv(W, s_y)

#step 4.
alngle_histogram = np.ones(36)#36bins
# for i in range(patch_size):
#     for j in range(patch_size):
#         angle = XX
#         alngle_histogram[angle] +=1

#step 5.
dominant_angle = max(alngle_histogram)#degree (not radian)


#step 6. do rotation normalization
rot_matrix = cv.getRotationMatrix2D((patch_size/2, patch_size/2), -dominant_angle, 1)
rot_W = cv.warpAffine(W, rot_matrix, (patch_size, patch_size))


#step 7. extract 16x16 window
w = rot_W[patch_size_half-8 : patch_size_half+8, patch_size_half-8 : patch_size_half+8]#16x16 window w

# --- Step 8 & 9: 그래디언트 계산 및 16개 셀 HOG 집계 ---

# 8.1: 'w' (16x16) 전체의 x, y 그래디언트(경사도)를 계산
# (ksize=1은 [-1, 0, 1] 커널과 유사하게 간단한 차이를 계산)
sobelx = cv.Sobel(w, cv.CV_64F, 1, 0, ksize=1)
sobely = cv.Sobel(w, cv.CV_64F, 0, 1, ksize=1)

# 8.2: 각 픽셀의 그래디언트 크기(magnitude)와 방향(orientation) 계산
magnitude = cv.magnitude(sobelx, sobely)
orientation = cv.phase(sobelx, sobely, angleInDegrees=True) # 0~360도

# 8.3: (SIFT 핵심) 16x16 윈도우 중심부에 가중치를 주는 가우시안 마스크 생성
# (sigma=8은 16x16 윈도우 크기의 절반)
gauss_y = cv.getGaussianKernel(16, 8.0)
gauss_x = cv.getGaussianKernel(16, 8.0)
gauss_weight = gauss_y * gauss_x.T # 16x16 가중치 맵

# 8.4: 그래디언트 크기에 가우시안 가중치 적용
weighted_magnitude = magnitude * gauss_weight

# 9.1: 16개 셀의 8방향 HOG 벡터(총 16개)를 저장할 리스트
hog = []
cell_size = 4
num_cells = 4
num_bins = 8      # SIFT는 8방향 히스토그램 사용 (0, 45, 90...)
bin_width = 360 / num_bins # 45.0도

# 9.2: 16x16 윈도우를 4x4 셀 그리드로 순회
for i in range(num_cells): # y방향 셀 (0~3)
    for j in range(num_cells): # x방향 셀 (0~3)
        
        # 9.3: 이 셀(i, j)을 위한 8방향 히스토그램 초기화
        hog_8d_vector = np.zeros(num_bins)
        
        # 9.4: 현재 셀의 4x4 영역 좌표 계산
        y_start, y_end = i * cell_size, (i + 1) * cell_size
        x_start, x_end = j * cell_size, (j + 1) * cell_size
        
        # 9.5: 해당 영역의 가중 크기(mag)와 방향(ori) 추출
        cell_w_mag = weighted_magnitude[y_start:y_end, x_start:x_end]
        cell_ori = orientation[y_start:y_end, x_start:x_end]
        
        # 9.6: 셀 내부 16개 픽셀(4x4)을 순회
        for y_pix in range(cell_size):
            for x_pix in range(cell_size):
                
                mag_val = cell_w_mag[y_pix, x_pix]
                ori_val = cell_ori[y_pix, x_pix]
                
                # 9.7: (SIFT 핵심) 선형 보간법(Linear Interpolation)
                # 픽셀의 방향이 정확히 45도, 90도에 떨어지지 않기 때문에,
                # 가장 가까운 두 개의 bin에 비례하여 값을 나눠서 더함 (안정성 증가)
                
                bin_pos = ori_val / bin_width # (예: 60도 -> 60/45 = 1.33)
                
                bin_idx_1 = int(bin_pos) % num_bins     # (예: bin 1 (45도))
                bin_idx_2 = (bin_idx_1 + 1) % num_bins # (예: bin 2 (90도))
                
                weight_2 = bin_pos - int(bin_pos) # (예: 0.33) -> bin 2에 줄 가중치
                weight_1 = 1.0 - weight_2         # (예: 0.67) -> bin 1에 줄 가중치
                
                # 9.8: 두 bin에 가중치를 적용하여 magnitude를 더함
                hog_8d_vector[bin_idx_1] += mag_val * weight_1
                hog_8d_vector[bin_idx_2] += mag_val * weight_2
        
        # 9.9: 이 셀의 8방향 HOG 벡터를 리스트에 추가
        hog.append(hog_8d_vector)

# 'hog' 리스트에는 16개의 (8차원) HOG 벡터가 저장됨

# --- Step 10: 128차원 기술자 생성 및 정규화 ---

# 10.1: 16개의 8차원 벡터를 하나의 128차원 리스트로 펼치기
sift_dscr1 = []
for hog_vector in hog:
    sift_dscr1.extend(hog_vector)

# 10.2: (SIFT 핵심) 조명 불변성을 위한 L2 정규화
sift_dscr1 = np.array(sift_dscr1)
norm = np.linalg.norm(sift_dscr1)
if norm == 0:
    # (드물게 모든 그래디언트가 0일 경우, 0으로 나눔 방지)
    sift_dscr1_norm = sift_dscr1
else:
    sift_dscr1_norm = sift_dscr1 / norm

# 10.3: (SIFT 핵심) 클리핑 (Clipping)
# 특정 방향의 그래디언트가 너무 강하게 영향을 주는 것을 막기 위해
# 벡터의 모든 값을 0.2 이하로 제한
sift_dscr1_clipped = np.clip(sift_dscr1_norm, 0, 0.2)

# 10.4: (SIFT 핵심) 재정규화
# 클리핑 후 벡터의 총합이 1이 아닐 수 있으므로 L2 정규화
norm_final = np.linalg.norm(sift_dscr1_clipped)
if norm_final == 0:
    sift_dscr1_final = sift_dscr1_clipped
else:
    sift_dscr1_final = sift_dscr1_clipped / norm_final

(322, 314)


In [41]:
print(sift_dscr1_final.shape)

(128,)


In [44]:
# --- 1. 주 방향성 계산 함수 (비어있던 Step 3-5 구현) ---
# 23x23 패치를 받아 주 방향(각도)을 반환
def calculate_dominant_angle(patch_W):
    
    # 3.1: 그래디언트 계산
    sobelx = cv.Sobel(patch_W, cv.CV_64F, 1, 0, ksize=3) # ksize=3 사용 (더 안정적)
    sobely = cv.Sobel(patch_W, cv.CV_64F, 0, 1, ksize=3)
    
    # 3.2: 크기(magnitude)와 방향(orientation) 계산
    magnitude = cv.magnitude(sobelx, sobely)
    orientation = cv.phase(sobelx, sobely, angleInDegrees=True)

    # 3.3: 패치 중심에 가중치를 주는 가우시안 웨이트 (SIFT 방식)
    patch_size = patch_W.shape[0]
    patch_size_half = patch_size // 2
    sigma = patch_size * 0.5 # SIFT 논문에서는 1.5 * keypoint scale
    
    gauss_y = cv.getGaussianKernel(patch_size, sigma)
    gauss_x = cv.getGaussianKernel(patch_size, sigma)
    gauss_weight = gauss_y * gauss_x.T
    
    weighted_magnitude = magnitude * gauss_weight

    # 4. 36-bin 히스토그램 생성 (10도 단위)
    alngle_histogram = np.zeros(36)
    bin_width = 10.0 # 360 / 36

    for y in range(patch_size):
        for x in range(patch_size):
            mag_val = weighted_magnitude[y, x]
            ori_val = orientation[y, x]
            
            # (간단한 집계: 보간법 생략)
            bin_idx = int(ori_val / bin_width) % 36
            alngle_histogram[bin_idx] += mag_val

    # 5. 주 방향(Dominant Angle) 결정
    # max()가 아닌 argmax()로 가장 큰 값의 '인덱스'를 찾아야 함
    dominant_bin = np.argmax(alngle_histogram)
    dominant_angle = dominant_bin * bin_width # (예: 인덱스 10 -> 100도)
    
    return dominant_angle


# --- 2. SIFT 기술자 계산 함수 (Step 2 ~ 10 통합) ---
# (x, y) 좌표 1개를 받아 128차원 기술자 1개를 반환
def compute_sift_descriptor(gray_img, feat_x, feat_y):
    
    # (Step 2. 패치 추출)
    patch_size = 23
    patch_size_half = patch_size // 2
    # [참고] 원본 gray (0~255)에서 추출 후, 함수 내부에서 0~1로 정규화
    W_u8 = gray[feat_y-patch_size_half:feat_y+patch_size_half+1, 
                feat_x-patch_size_half:feat_x+patch_size_half+1]
    W = W_u8 / 255.0 # (0~1)
    
    # (Step 3-5. 주 방향성 계산)
    # 위에서 구현한 함수 호출 (0~255 스케일의 패치로 계산하는 것이 더 안정적일 수 있음)
    dominant_angle = calculate_dominant_angle(W_u8.astype(np.float32))

    # (Step 6. 회전 정규화)
    center = (patch_size_half, patch_size_half) # (중심점 11, 11)
    rot_matrix = cv.getRotationMatrix2D(center, -dominant_angle, 1)
    rot_W = cv.warpAffine(W, rot_matrix, (patch_size, patch_size))

    # (Step 7. 16x16 윈도우 추출)
    w = rot_W[patch_size_half-8 : patch_size_half+8, 
              patch_size_half-8 : patch_size_half+8] # 16x16

    # (Step 8 & 9: HOG 집계)
    # (이전 단계에서 완성한 코드를 그대로 사용)
    sobelx = cv.Sobel(w, cv.CV_64F, 1, 0, ksize=1)
    sobely = cv.Sobel(w, cv.CV_64F, 0, 1, ksize=1)
    magnitude = cv.magnitude(sobelx, sobely)
    orientation = cv.phase(sobelx, sobely, angleInDegrees=True)

    gauss_y = cv.getGaussianKernel(16, 8.0)
    gauss_x = cv.getGaussianKernel(16, 8.0)
    gauss_weight = gauss_y * gauss_x.T
    weighted_magnitude = magnitude * gauss_weight

    hog = []
    cell_size = 4
    num_cells = 4
    num_bins = 8
    bin_width = 360 / num_bins

    for i in range(num_cells):
        for j in range(num_cells):
            hog_8d_vector = np.zeros(num_bins)
            y_start, y_end = i * cell_size, (i + 1) * cell_size
            x_start, x_end = j * cell_size, (j + 1) * cell_size
            
            cell_w_mag = weighted_magnitude[y_start:y_end, x_start:x_end]
            cell_ori = orientation[y_start:y_end, x_start:x_end]
            
            for y_pix in range(cell_size):
                for x_pix in range(cell_size):
                    mag_val = cell_w_mag[y_pix, x_pix]
                    ori_val = cell_ori[y_pix, x_pix]
                    
                    bin_pos = ori_val / bin_width
                    bin_idx_1 = int(bin_pos) % num_bins
                    bin_idx_2 = (bin_idx_1 + 1) % num_bins
                    weight_2 = bin_pos - int(bin_pos)
                    weight_1 = 1.0 - weight_2
                    
                    hog_8d_vector[bin_idx_1] += mag_val * weight_1
                    hog_8d_vector[bin_idx_2] += mag_val * weight_2
            
            hog.append(hog_8d_vector)

    # (Step 10. 기술자 생성 및 정규화)
    sift_dscr1 = []
    for hog_vector in hog:
        sift_dscr1.extend(hog_vector)

    sift_dscr1 = np.array(sift_dscr1)
    norm = np.linalg.norm(sift_dscr1)
    sift_dscr1_norm = sift_dscr1 / norm if norm != 0 else sift_dscr1

    sift_dscr1_clipped = np.clip(sift_dscr1_norm, 0, 0.2)
    
    norm_final = np.linalg.norm(sift_dscr1_clipped)
    sift_dscr1_final = sift_dscr1_clipped / norm_final if norm_final != 0 else sift_dscr1_clipped

    return sift_dscr1_final

# --- 3. 메인 코드 실행 ---

# (Harris 코너 실행 부분 - 원본과 동일)
img = cv.imread("frame10.png")
gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)
gray = np.float32(gray)
dst = cv.cornerHarris(gray,2,3,0.04)

(img_h, img_w) = dst.shape
corner_thr = 0.01*dst.max()

features1 = []
for i in range(0, img_h):
    for j in range(0, img_w):
        if dst[i][j] > corner_thr:
            features1.append((j,i)) # (x, y)

# (Step 1. 경계 필터링 - 원본과 동일)
patch_size = 23
patch_size_half = patch_size // 2

valid_features1 = [(x, y) for (x, y) in features1 
                   if (x >= patch_size_half and 
                       x < img_w - patch_size_half and 
                       y >= patch_size_half and 
                       y < img_h - patch_size_half)]

# --- ⬇️ 여기부터 수정 ⬇️ ---

# 결과를 저장할 리스트
all_keypoints1 = []     # (x, y) 좌표 저장
all_descriptors1 = []   # 128차원 기술자 저장

print(f"총 {len(valid_features1)}개의 유효한 코너에 대해 기술자를 계산합니다...")

# (random.choice 대신 for 루프 사용)
for (feat_x, feat_y) in valid_features1:
    
    # 2. 위에서 만든 함수를 호출하여 기술자 계산
    descriptor = compute_sift_descriptor(gray, feat_x, feat_y)
    
    # 3. 결과 저장
    all_keypoints1.append((feat_x, feat_y))
    all_descriptors1.append(descriptor)

# --- ⬆️ 수정 끝 ⬆️ ---

# 결과 확인
print(f"총 {len(all_descriptors1)}개의 기술자가 생성되었습니다.")
# all_descriptors는 N x 128 형태의 리스트가 됩니다.
# (NumPy 배열로 변환)
all_descriptors_np1 = np.array(all_descriptors1)
print(f"최종 기술자 배열 형태: {all_descriptors_np1.shape}")

총 1494개의 유효한 코너에 대해 기술자를 계산합니다...
총 1494개의 기술자가 생성되었습니다.
최종 기술자 배열 형태: (1494, 128)


In [43]:
# --- 3. 메인 코드 실행 ---

# (Harris 코너 실행 부분 - 원본과 동일)
img = cv.imread("frame11.png")
gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)
gray = np.float32(gray)
dst = cv.cornerHarris(gray,2,3,0.04)

(img_h, img_w) = dst.shape
corner_thr = 0.01*dst.max()

features2 = []
for i in range(0, img_h):
    for j in range(0, img_w):
        if dst[i][j] > corner_thr:
            features2.append((j,i)) # (x, y)

# (Step 1. 경계 필터링 - 원본과 동일)
patch_size = 23
patch_size_half = patch_size // 2

valid_features2 = [(x, y) for (x, y) in features2
                   if (x >= patch_size_half and 
                       x < img_w - patch_size_half and 
                       y >= patch_size_half and 
                       y < img_h - patch_size_half)]

# --- ⬇️ 여기부터 수정 ⬇️ ---

# 결과를 저장할 리스트
all_keypoints2 = []     # (x, y) 좌표 저장
all_descriptors2 = []   # 128차원 기술자 저장

print(f"총 {len(valid_features2)}개의 유효한 코너에 대해 기술자를 계산합니다...")

# (random.choice 대신 for 루프 사용)
for (feat_x, feat_y) in valid_features2:
    
    # 2. 위에서 만든 함수를 호출하여 기술자 계산
    descriptor = compute_sift_descriptor(gray, feat_x, feat_y)
    
    # 3. 결과 저장
    all_keypoints2.append((feat_x, feat_y))
    all_descriptors2.append(descriptor)

# --- ⬆️ 수정 끝 ⬆️ ---

# 결과 확인
print(f"총 {len(all_descriptors2)}개의 기술자가 생성되었습니다.")
# all_descriptors는 N x 128 형태의 리스트가 됩니다.
# (NumPy 배열로 변환)
all_descriptors_np2 = np.array(all_descriptors2)
print(f"최종 기술자 배열 형태: {all_descriptors_np2.shape}")

총 1457개의 유효한 코너에 대해 기술자를 계산합니다...
총 1457개의 기술자가 생성되었습니다.
최종 기술자 배열 형태: (1457, 128)


In [50]:
img1 = cv.imread('frame10.png') 
img2 = cv.imread('frame11.png')


all_descriptors_np1 = all_descriptors_np1.astype(np.float32)
all_descriptors_np2 = all_descriptors_np2.astype(np.float32)


kp1 = [cv.KeyPoint(x, y, 1) for (x, y) in all_keypoints1]
kp2 = [cv.KeyPoint(x, y, 1) for (x, y) in all_keypoints2]

bf = cv.BFMatcher(cv.NORM_L2, crossCheck=False)
matches = bf.knnMatch(all_descriptors_np1, all_descriptors_np2, k=2)

good_matches = []
ratio_thresh = 0.3

for m, n in matches:
    # 1번째 이웃의 거리(m.distance)가
    # 2번째 이웃의 거리(n.distance)의 75%보다 작으면 좋은 매칭으로 간주
    if m.distance < ratio_thresh * n.distance:
        # cv.drawMatchesKnn는 리스트의 리스트 형태[[m], [m]...]를 입력받음
        good_matches.append([m])


print(f"총 매칭 수: {len(matches)}, 유효한 매칭 수 (Ratio Test): {len(good_matches)}")

총 매칭 수: 1494, 유효한 매칭 수 (Ratio Test): 58


In [51]:
img_matches = cv.drawMatchesKnn(
    img1, kp1, 
    img2, kp2, 
    good_matches, 
    None, 
    flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

# (선택) 결과 이미지가 너무 클 수 있으므로 리사이즈
(h, w) = img_matches.shape[:2]
if w > 1920: # 가로가 1920보다 크면 절반으로 줄임 (값 조절 가능)
    img_matches = cv.resize(img_matches, (w // 2, h // 2), interpolation=cv.INTER_AREA)

cv.imshow('Matches', img_matches)
cv.waitKey(0)
cv.destroyAllWindows()